In [1]:
using DataFrames, Distributions, CSV, Turing, Optim, StatsBase, StatsFuns, Random

Random.seed!(42)

# Read the vitamin D dataset
data = CSV.read("../../data/vitd/nhanes.csv", DataFrame)
data[!, :"RIAGENDR"] = data[!, :"RIAGENDR"] .- 1
data[!, :"DPQ020"] = data[!, :"DPQ020"] .+ 1

# Look at the first few rows of the data
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,27.8,0.0,62.0,76.1,4.14,1.0,3.0
2,30.8,0.0,53.0,56.5,0.0,1.0,1.0
3,28.8,0.0,78.0,87.5,1.81,1.0,3.0
4,28.0,0.0,22.0,47.2,2.98,1.0,2.0
5,27.6,0.0,46.0,44.5,1.73,1.0,1.0


In [2]:
#= Convert categorical data to integers =#

data[!, :"RIAGENDR"] = convert(Array{Int}, data[!, :"RIAGENDR"])
data[!, :"DPQ020"] = convert(Array{Int}, data[!, "DPQ020"])
data[!, :"SMQ040"] = convert(Array{Int}, data[!, :"SMQ040"]);

# Get variables
bmi = data[!, :"BMXBMI"]
gender = data[!, :"RIAGENDR"]
age = data[!, :"RIDAGEYR"]
vitd = data[!, :"LBXVIDMS"]
poverty = data[!, :"INDFMMPI"]
depression = data[!, :"DPQ020"]
smoking = data[!, :"SMQ040"];

In [3]:
# Turing Model Optimized by Optim.jl

@model function depression_model_fitted(bmi, gender, age, vitd, poverty, depression, smoking)
    # Define the length 
    n = min(length(bmi), length(gender), length(age), length(vitd), length(poverty), length(depression), length(smoking))

    for i in 1:n
        dist1_age = Normal(31.995, 7.2)
        dist2_age = Normal(51.1811, 8.7)
        dist1_pov = Normal(1, 2)
        dist2_pov = Normal(5, 0.001)
        bmi[i] ~ LogNormal(3.32504, 1.40692e-9)
        gender[i] ~ Bernoulli(0.4)
        age[i] ~ MixtureModel([dist1_age, dist2_age], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53])
        poverty[i] ~ MixtureModel([dist1_pov, dist2_pov], [0.5, 0.5])
        vitd[i] ~ LogNormal(4.07 + 1.01 * smoking[i] + 0.738 * age[i]
        + 1.217 * gender[i] + -1.743 * bmi[i], 1.20779e-8)
        
        linear_predictor = -0.125589 + vitd[i] * 0.043 +
                            bmi[i] * -0.801 + age[i] * 0.216 +
                            poverty[i] * -0.677474 + gender[i] * 0.439184 +
                            smoking[i] * 3.15499

        depression[i] ~ Normal(linear_predictor, 0.01)
    end
end


depression_model_fitted (generic function with 2 methods)

In [4]:
model_depr = depression_model_fitted(bmi, gender, age, vitd, poverty, repeat([missing], length(bmi)), smoking)

chain_depr = sample(model_depr, PG(10), 100)

Sampling   0%|                                          |  ETA: N/A
Sampling   1%|▍                                         |  ETA: 0:21:17
Sampling   2%|▉                                         |  ETA: 0:18:16
Sampling   3%|█▎                                        |  ETA: 0:16:57
Sampling   4%|█▋                                        |  ETA: 0:16:14
Sampling   5%|██▏                                       |  ETA: 0:15:45
Sampling   6%|██▌                                       |  ETA: 0:15:22
Sampling   7%|███                                       |  ETA: 0:15:04
Sampling   8%|███▍                                      |  ETA: 0:14:55
Sampling   9%|███▊                                      |  ETA: 0:14:38
Sampling  10%|████▎                                     |  ETA: 0:14:23
Sampling  11%|████▋                                     |  ETA: 0:14:09
Sampling  12%|█████                                     |  ETA: 0:13:57
Sampling  13%|█████▌                                    |  ETA: 0:13

Chains MCMC chain (100×1630×1 Array{Float64, 3}):

Log evidence      = -2.4957112189096795e21
Iterations        = 1:1:100
Number of chains  = 1
Samples per chain = 100
Wall duration     = 923.53 seconds
Compute duration  = 923.53 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47],

In [5]:
depression_days = Array{Int}(undef, length(depression))

for i in 1:length(depression)
    depression_days[i] = round(mean(chain_depr["depression[$i]"]))
end

depression_days = clamp.(depression_days, 1, 4)

countmap(depression_days)

Dict{Int64, Int64} with 4 entries:
  4 => 193
  2 => 61
  3 => 50
  1 => 1324

In [6]:
total = 0
for i in 1:length(depression)
    is_correct = depression_days[i] == depression[i]
    total += is_correct
end
good_rat = total/ length(depression)

0.5872235872235873

In [7]:
model_depr = depression_model_fitted(bmi, gender, age, vitd .+ 20, poverty, repeat([missing], length(bmi)), smoking)

chain_depr = sample(model_depr, PG(10), 100)

Sampling   0%|                                          |  ETA: N/A
Sampling   1%|▍                                         |  ETA: 0:16:35
Sampling   2%|▉                                         |  ETA: 0:15:57
Sampling   3%|█▎                                        |  ETA: 0:15:36
Sampling   4%|█▋                                        |  ETA: 0:15:13
Sampling   5%|██▏                                       |  ETA: 0:15:04
Sampling   6%|██▌                                       |  ETA: 0:14:50
Sampling   7%|███                                       |  ETA: 0:14:37
Sampling   8%|███▍                                      |  ETA: 0:14:24
Sampling   9%|███▊                                      |  ETA: 0:14:11
Sampling  10%|████▎                                     |  ETA: 0:14:00
Sampling  11%|████▋                                     |  ETA: 0:13:50
Sampling  12%|█████                                     |  ETA: 0:13:41
Sampling  13%|█████▌                                    |  ETA: 0:13

Chains MCMC chain (100×1630×1 Array{Float64, 3}):

Log evidence      = -2.5451828860457756e21
Iterations        = 1:1:100
Number of chains  = 1
Samples per chain = 100
Wall duration     = 952.15 seconds
Compute duration  = 952.15 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47],

In [10]:
depression_days = Array{Int}(undef, length(depression))

for i in 1:length(depression)
    depression_days[i] = round(mean(chain_depr["depression[$i]"]))
end

depression_days = clamp.(depression_days, 1, 4)

countmap(depression_days)

Dict{Int64, Int64} with 4 entries:
  4 => 236
  2 => 90
  3 => 55
  1 => 1247